In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    if filenames:
        print("   ", filenames[:5])

In [ ]:
import tensorflow as tf
from tensorflow import keras

# ============================
# LOAD TRAINING DATA
# ============================

train_ds = keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/salader/dogsvscats/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256),
    shuffle=True
)

# ============================
# LOAD TEST / VALIDATION DATA
# ============================

validation_ds = keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/salader/dogsvscats/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(256, 256),
    shuffle=False
)

In [ ]:
def process(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = train_ds.map(process)
validation_ds = validation_ds.map(process)

In [ ]:
from keras.models import Sequential
from keras.layers import (
    Dense, Conv2D, MaxPooling2D,
    BatchNormalization, Dropout,
    GlobalAveragePooling2D
)

model = Sequential()

# Input
model.add(keras.Input(shape=(256, 256, 3)))

# Block 1
model.add(Conv2D(32, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Block 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Block 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(2,2))

# Instead of Flatten()
model.add(GlobalAveragePooling2D())

# Fully connected
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    epochs=10,
    validation_data=validation_ds
)

In [ ]:
# ==========================================
# 6. EVALUATION (LOSS & ACCURACY PLOTS)
# ==========================================

import matplotlib.pyplot as plt

# Plot Accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


# ==========================================
# 7. INFERENCE (PREDICTING ON A NEW IMAGE)
# ==========================================

import cv2
import os
import numpy as np

# Change this to any image from your Kaggle dataset
image_path = '/kaggle/input/datasets/piyush614/dog-file/Screenshot 2026-09-16 155253.png'


if os.path.exists(image_path):

    # Read image
    test_img = cv2.imread(image_path)

    # Convert BGR → RGB
    test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)

    # Display image
    plt.figure(figsize=(5, 5))
    plt.imshow(test_img_rgb)
    plt.title("Input Image")
    plt.axis('off')
    plt.show()

    # Resize image
    test_img_resized = cv2.resize(test_img_rgb, (256, 256))

    # Convert to float and normalize
    test_input = test_img_resized.astype('float32') / 255.0

    # Add batch dimension
    test_input = np.expand_dims(test_input, axis=0)

    # Prediction
    prediction = model.predict(test_input, verbose=0)

    probability = prediction[0][0]

    print(f"Raw Prediction Value: {probability:.4f}")

    # Binary classification
    if probability < 0.5:
        print("Prediction: CAT 🐱")
    else:
        print("Prediction: DOG 🐶")

else:
    print(f"Image not found at: {image_path}")